# Teste de Extração: PIB VAB Agropecuária
Extrai o Valor Adicionado Bruto da Agropecuária (Tabela 5938) via API SIDRA.

In [ ]:
import pandas as pd
import sidrapy
import time
from pathlib import Path
from tqdm import tqdm

DATA_DIR = Path("data")
BRONZE_DIR = DATA_DIR / "bronze"
BRONZE_DIR.mkdir(parents=True, exist_ok=True)

def baixar_pib(ano):
    try:
        # Variável 513 = VAB Agropecuária
        df = sidrapy.get_table(table_code="5938", territorial_level="6", ibge_territorial_code="all", period=str(ano), variable="513")
        return df.iloc[1:].copy() if df is not None and len(df) > 1 else pd.DataFrame()
    except Exception as e:
        print(f"Erro PIB {ano}: {e}")
        return pd.DataFrame()

anos = [2020, 2021]
for ano in tqdm(anos):
    df = baixar_pib(ano)
    if not df.empty:
        out = BRONZE_DIR / "pib_vab_agro" / f"ano={ano}"
        out.mkdir(parents=True, exist_ok=True)
        df.to_parquet(out / "dados.parquet")
    time.sleep(1)
print("Finalizado!")